In [2]:
import ee
ee.Authenticate()
ee.Initialize()
import pandas as pd

In [3]:
TREE = 10
SHRUB = 20
GRASS = 30
CROP = 40
URBAN = 50
BARE = 60
SNOW = 70
WATER = 80
WETLAND = 90
MANGROVES = 95
MOSS = 100

WORLD_COVER_LABELS = {
    10: "Tree cover",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up",
    60: "Bare / sparse vegetation",
    70: "Snow and ice",
    80: "Permanent water bodies",
    90: "Herbaceous wetland",
    95: "Mangroves",
    100: "Moss and lichen"
}

In [4]:
def get_landcover_stats(lat, lon, radius_m=10000, scale=10):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(radius_m)

    worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')

    histogram_dict = worldcover.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9
    ).getInfo() or {}

    histogram = histogram_dict.get('Map', {}) or {}
    histogram = {int(k): int(v) for k, v in histogram.items()}

    total_pixels = sum(histogram.values()) if histogram else 0

    percentages = {}
    for cls in WORLD_COVER_LABELS:
        percentages[cls] = (histogram.get(cls, 0) / total_pixels * 100.0) if total_pixels > 0 else 0.0

    forest_pct = percentages.get(TREE, 0) + percentages.get(MANGROVES, 0)
    agriculture_pct = percentages.get(CROP, 0)
    urban_pct = percentages.get(URBAN, 0)
    bare_pct = percentages.get(BARE, 0)
    snow_pct = percentages.get(SNOW, 0)
    water_pct = percentages.get(WATER, 0)
    wetland_pct = percentages.get(WETLAND, 0)
    open_vegetation_pct = percentages.get(SHRUB, 0) + percentages.get(GRASS, 0) + percentages.get(MOSS, 0)

    vegetation_pct = forest_pct + agriculture_pct + open_vegetation_pct

    return {
        "point": point,
        "region": region,
        "worldcover": worldcover,
        "histogram": histogram,
        "percentages": percentages,
        "forest_pct": forest_pct,
        "agriculture_pct": agriculture_pct,
        "urban_pct": urban_pct,
        "bare_pct": bare_pct,
        "snow_pct": snow_pct,
        "water_pct": water_pct,
        "wetland_pct": wetland_pct,
        "open_vegetation_pct": open_vegetation_pct,
        "vegetation_pct": vegetation_pct
    }

In [5]:
def aggregate_landcover(percentages, wetlands_as_water=True):
    water_pct = percentages.get(80, 0.0)
    if wetlands_as_water:
        water_pct += percentages.get(90, 0.0)

    forest_pct = percentages.get(10, 0.0) + percentages.get(95, 0.0)
    vegetation_pct = (
        percentages.get(20, 0.0) +
        percentages.get(30, 0.0) +
        percentages.get(100, 0.0)
    )

    aggregated = {
        "water_pct": water_pct,
        "urban_pct": percentages.get(50, 0.0),
        "forest_pct": forest_pct,
        "agriculture_pct": percentages.get(40, 0.0),
        "bare_pct": percentages.get(60, 0.0),
        "vegetation_pct": vegetation_pct,
        "snow_pct": percentages.get(70, 0.0),
    }

    return aggregated

In [6]:
def classify_terrain(mean_elev, mean_slope, elev_range):
    if mean_elev >= 1000 or (mean_slope >= 8 and elev_range >= 300):
        return 'Mountainous'
    if mean_elev >= 800 and mean_slope < 5 and elev_range < 250:
        return 'Plateau'
    if mean_elev >= 200 and (mean_slope >= 5 or elev_range >= 150):
        return 'Hilly'
    return 'Plain'


def get_terrain_stats(lat, lon, radius_m=5000, scale=90):
    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(radius_m)

    srtm = ee.Image('USGS/SRTMGL1_003').select('elevation')
    slope = ee.Terrain.slope(srtm)

    elev_stats = srtm.reduceRegion(
        reducer=ee.Reducer.mean()
            .combine(ee.Reducer.stdDev(), sharedInputs=True)
            .combine(ee.Reducer.minMax(), sharedInputs=True),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9
    ).getInfo() or {}

    slope_stats = slope.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9
    ).getInfo() or {}

    mean_elev = float(elev_stats.get('elevation_mean', 0) or 0)
    std_elev = float(elev_stats.get('elevation_stdDev', 0) or 0)
    min_elev = float(elev_stats.get('elevation_min', 0) or 0)
    max_elev = float(elev_stats.get('elevation_max', 0) or 0)
    elev_range = max_elev - min_elev
    mean_slope = float(slope_stats.get('slope', 0) or 0)

    terrain_type = classify_terrain(mean_elev, mean_slope, elev_range)
    is_mountainous = terrain_type in ('Mountainous', 'mountain', 'Plateau')

    return {
        "mean_elevation": mean_elev,
        "std_elevation": std_elev,
        "min_elevation": min_elev,
        "max_elevation": max_elev,
        "elev_range": elev_range,
        "mean_slope": mean_slope,
        "terrain_type": terrain_type,
        "is_mountainous": is_mountainous
    }

In [7]:
def get_distance_to_water_km(point, worldcover, max_distance_km=50, scale=30):
    water_mask = worldcover.eq(WATER).rename('water')

    distance_transform = water_mask.fastDistanceTransform().sqrt() \
        .multiply(ee.Image.pixelArea().sqrt())

    dist_to_water = distance_transform.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=100
    ).getInfo().get('distance', float('inf'))

    dist_to_water_km = dist_to_water / 1000 if dist_to_water != float('inf') else 999

    if dist_to_water_km is None:
        return float(max_distance_km)

    return dist_to_water_km

In [8]:
def classify_scene(percentages, forest_pct, agriculture_pct, urban_pct,
                   bare_pct, snow_pct, water_pct, wetland_pct,
                   open_vegetation_pct, vegetation_pct,
                   is_mountainous, elev_range,
                   dist_to_water_km):
    
    is_coastal = dist_to_water_km < 10
    
    def make_result(scene_type, confidence, reason, dominant_pct=None):
        return {
            "scene_type": scene_type,
            "confidence": confidence,
            "dominant_pct": dominant_pct,
            "is_coastal": is_coastal,
            "reason": reason,
            "dist_to_water_km": dist_to_water_km
        }
    
    # Priority 1: Water dominance (>80%)
    if water_pct > 80:
        return make_result(
            "Ocean",
            "High",
            f"{water_pct:.1f}% water coverage",
            water_pct
        )
    
    # Priority 2: Coastal (water 25-70% OR near water)
    if (22 < water_pct <= 70) or (is_coastal and water_pct > 22):
        return make_result(
            "Coastline",
            "High",
            f"{water_pct:.1f}% water and {dist_to_water_km:.1f}km from coast",
            water_pct
        )
    
    # Priority 3: Mountainous terrain (>200m elevation range)
    if is_mountainous and elev_range > 200:
        # Determine mountain subtype
        if bare_pct > 60:
            subtype = " (Desert)"
        elif forest_pct > 60:
            subtype = " (Forested)"
        else:
            subtype = ""
        
        return make_result(
            "Mountain",
            "High",
            f"{elev_range:.0f}m elevation range{subtype}",
            None
        )
    
    # Priority 4: Snow/Ice (>30% snow)
    if snow_pct > 30:
        return make_result(
            "Snow/Ice",
            "High",
            f"{snow_pct:.1f}% snow coverage",
            snow_pct
        )
    
    # Priority 5: Urban dominance (>30%)
    if urban_pct > 30:
        return make_result(
            "Urban",
            "High",
            f"{urban_pct:.1f}% urban coverage",
            urban_pct
        )
    
    # Priority 6: Dominant land cover type
    land_cover = {
        "Forest": forest_pct,
        "Agriculture": agriculture_pct,
        "Desert": bare_pct,
        "Wetland": wetland_pct,
        "Urban": urban_pct
    }
    
    scene_type = max(land_cover, key=land_cover.get)
    dominant_pct = land_cover[scene_type]
    
    # Confidence based on dominance
    if dominant_pct >= 50:
        confidence = "High"
    elif dominant_pct >= 35:
        confidence = "Medium"
    else:
        confidence = "Low"
        scene_type = "Mixed"
    
    return make_result(
        scene_type,
        confidence,
        f"{dominant_pct:.1f}% {scene_type.lower()} coverage",
        dominant_pct
    )

In [9]:
def get_clear_reference_image(lat, lon, date='2024-12-08', search_days=730):
    """
    Find clearest Sentinel-2 image near your location
    Use this to determine scene type
    """
    point = ee.Geometry.Point([lon, lat])
    start = ee.Date(date).advance(-search_days, 'day')
    end = ee.Date(date).advance(search_days, 'day')
    
    collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(point) \
        .filterDate(start, end) \
        .sort('CLOUDY_PIXEL_PERCENTAGE')  # Clearest first
    
    if collection.size().getInfo() == 0:
        return None
    
    # Get clearest image
    clear_image = collection.first()
    cloud_pct = clear_image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
    
    print(f"Found Sentinel-2 with {cloud_pct}% clouds")
    
    # Get preview URL
    url = clear_image.getThumbURL({
        'bands': ['B4', 'B3', 'B2'],
        'min': 0,
        'max': 3000,
        'dimensions': 512,
        'region': point.buffer(20000).bounds()
    })
    
    return url

In [10]:
def classify_location(lat, lon, lc_radius_m=10000, terrain_radius_m=5000):
    lc = get_landcover_stats(lat, lon, radius_m=lc_radius_m, scale=100)
    terrain = get_terrain_stats(lat, lon, radius_m=terrain_radius_m, scale=90)
    dist_to_water_km = get_distance_to_water_km(lc["point"], lc["worldcover"], max_distance_km=50, scale=30)
    # link = get_clear_reference_image(lat, lon)
    link=""

    scene = classify_scene(
        percentages=lc["percentages"],
        forest_pct=lc["forest_pct"],
        agriculture_pct=lc["agriculture_pct"],
        urban_pct=lc["urban_pct"],
        bare_pct=lc["bare_pct"],
        snow_pct=lc["snow_pct"],
        water_pct=lc["water_pct"],
        wetland_pct=lc["wetland_pct"],
        open_vegetation_pct=lc["open_vegetation_pct"],
        vegetation_pct=lc["vegetation_pct"],
        is_mountainous=terrain["is_mountainous"],
        # terrain_type=terrain["terrain_type"],
        elev_range=terrain["elev_range"],
        dist_to_water_km=dist_to_water_km
    )

    return {
        "lat": lat,
        "lon": lon,
        "link": link,
        "scene_type": scene["scene_type"],
        "confidence": scene["confidence"],
        # "reason": scene["reason"],
        "is_coastal": scene["is_coastal"],
        "dist_to_water_km": scene["dist_to_water_km"],
        "terrain_type": terrain["terrain_type"],
        "terrain": terrain,
        "aggregated": {
            "water_pct": lc["water_pct"],
            "wetland_pct": lc["wetland_pct"],
            "urban_pct": lc["urban_pct"],
            "forest_pct": lc["forest_pct"],
            "agriculture_pct": lc["agriculture_pct"],
            "open_vegetation_pct": lc["open_vegetation_pct"],
            "bare_pct": lc["bare_pct"],
            "snow_pct": lc["snow_pct"],
            "vegetation_pct": lc["vegetation_pct"]
        },
        "raw_percentages": lc["percentages"],
        "histogram": lc["histogram"]
    }


In [15]:
CLOUD_PENALTIES = {
    "clear":    0.0,
    "light":    1.0,
    "moderate": 2.5,
    "heavy":    4.0,
    "overcast": 6.0,
}

TERRAIN_DIFFICULTY = {
    'Plain':       0.0,
    'Hilly':       1.0,
    'Mountainous': 2.0,
    'Plateau':     1.0,
}

SCENE_PENALTIES = {
    "Ocean":       4.0,
    "Desert":      2.5,
    "Mountain":    2.0,
    "Coastline":   1.5,
    "Mixed":       1.0,
    "Forest":      0.5,
    "Agriculture": 0.0,
    "Urban":       0.0,
}

DIFFICULTY_THRESHOLDS = [
    (6.5, "Extreme"),
    (5.0, "Hard"),
    (3.0, "Medium"),
    (0.0, "Easy"),
]


def categorize_cloud_pct(cloud_pct: float) -> str:
    """Categorize cloud cover percentage into broad bins."""
    cloud_pct = max(0.0, min(100.0, float(cloud_pct)))
    if cloud_pct <= 10:
        return "clear"
    if cloud_pct <= 30:
        return "light"
    if cloud_pct <= 60:
        return "moderate"
    if cloud_pct <= 85:
        return "heavy"
    return "overcast"


def classify_difficulty_complete(row) -> str:
    """
    Classify georeferencing difficulty from a dataset row.

    Challenge score components:
        Cloud cover   : 0.0 – 6.0  (dominant factor)
        Terrain       : 0.0 – 2.0
        Scene type    : 0.0 – 4.0
        Feature poverty: 0.0 – 1.5

    Thresholds:
        Extreme : >= 6.5
        Hard    : >= 5.0
        Medium  : >= 3.0
        Easy    : <  3.0
    """
    # cloud_pct   = float(row['cloud_pct'])
    terrain     = row['terrain']
    scene_type  = row['scene_type']
    water_pct   = float(row['water_pct'])
    bare_pct    = float(row['bare_pct'])
    snow_pct    = float(row['snow_pct'])

    challenge = 0.0

    # Cloud cover penalty
    challenge += CLOUD_PENALTIES[row["cloud_category"]]

    # Terrain penalty
    challenge += TERRAIN_DIFFICULTY.get(terrain, 0.0)

    # Scene inherent difficulty
    challenge += SCENE_PENALTIES.get(scene_type, 1.0)

    # Feature poverty (water + bare + snow = low texture)
    feature_poor = water_pct + bare_pct + snow_pct
    if feature_poor > 70:
        challenge += 1.5
    elif feature_poor > 50:
        challenge += 0.5

    # Map score to difficulty label
    for threshold, label in DIFFICULTY_THRESHOLDS:
        if challenge >= threshold:
            return label

    return "Easy"

In [22]:
df = pd.read_csv("dataset_raw.csv")

In [23]:
df

,image_id,lat,lon,cloud_category
0,PHISAT-2_L1_000001341_20241007065606_202410070...,20.3511,58.4490,clear
1,PHISAT-2_L1_000001389_20241019145238_202410191...,-31.8813,-69.4538,clear
2,PHISAT-2_L1_000001458_20241104065341_202411040...,20.5376,58.4834,clear
3,PHISAT-2_L1_000001466_20241107092209_202411070...,-23.2193,15.2696,clear
4,PHISAT-2_L1_000001654_20250225170746_202502251...,29.8323,-93.2903,overcast
...,...,...,...,...
185,PHISAT-2_L1_000004495_20260125053643_202601250...,26.4790,80.3402,clear
186,PHISAT-2_L1_000004521_20260127171741_202601271...,29.8954,-95.3844,clear
187,PHISAT-2_L1_000004551_20260131141318_202601311...,-26.1042,-56.5187,clear
188,PHISAT-2_L1_000004558_20260202161528_202602021...,26.5450,-80.4875,clear


In [25]:
for index, row in df.iterrows():
    results = classify_location(row['lat'], row['lon'])
    df.loc[index, "dist_to_water_km"] = results["dist_to_water_km"]
    aggregated = results["aggregated"]

    df.loc[index, "terrain"] = results["terrain_type"]
    df.loc[index, "scene_type"] = results["scene_type"]
    df.loc[index, "water_pct"] = aggregated["water_pct"]
    df.loc[index, "wetland_pct"] = aggregated["wetland_pct"]
    df.loc[index, "urban_pct"] = aggregated["urban_pct"]
    df.loc[index, "forest_pct"] = aggregated["forest_pct"]
    df.loc[index, "agriculture_pct"] = aggregated["agriculture_pct"]
    df.loc[index, "open_vegetation_pct"] = aggregated["open_vegetation_pct"]
    df.loc[index, "bare_pct"] = aggregated["bare_pct"]
    df.loc[index, "snow_pct"] = aggregated["snow_pct"]
    df.loc[index, "vegetation_pct"] = aggregated["vegetation_pct"]

    print(f"Processed {index+1}/{len(df)}: {results['scene_type']}")

Processed 1/190: Coastline
Processed 2/190: Desert
Processed 3/190: Desert
Processed 4/190: Desert
Processed 5/190: Coastline
Processed 6/190: Ocean
Processed 7/190: Mixed
Processed 8/190: Ocean
Processed 9/190: Forest
Processed 10/190: Forest
Processed 11/190: Ocean
Processed 12/190: Ocean
Processed 13/190: Urban
Processed 14/190: Urban
Processed 15/190: Agriculture
Processed 16/190: Urban
Processed 17/190: Ocean
Processed 18/190: Mixed
Processed 19/190: Ocean
Processed 20/190: Coastline
Processed 21/190: Forest
Processed 22/190: Mixed
Processed 23/190: Urban
Processed 24/190: Agriculture
Processed 25/190: Ocean
Processed 26/190: Desert
Processed 27/190: Coastline
Processed 28/190: Forest
Processed 29/190: Agriculture
Processed 30/190: Mountain
Processed 31/190: Forest
Processed 32/190: Agriculture
Processed 33/190: Mountain
Processed 34/190: Mountain
Processed 35/190: Agriculture
Processed 36/190: Mixed
Processed 37/190: Mixed
Processed 38/190: Agriculture
Processed 39/190: Mountain


In [26]:
for index, row in df.iterrows():
    print(f"Classifying difficulty for {index+1}/{len(df)}")
    
    difficulty = classify_difficulty_complete(row)
    df.loc[index, "difficulty"] = difficulty
    

Classifying difficulty for 1/190
Classifying difficulty for 2/190
Classifying difficulty for 3/190
Classifying difficulty for 4/190
Classifying difficulty for 5/190
Classifying difficulty for 6/190
Classifying difficulty for 7/190
Classifying difficulty for 8/190
Classifying difficulty for 9/190
Classifying difficulty for 10/190
Classifying difficulty for 11/190
Classifying difficulty for 12/190
Classifying difficulty for 13/190
Classifying difficulty for 14/190
Classifying difficulty for 15/190
Classifying difficulty for 16/190
Classifying difficulty for 17/190
Classifying difficulty for 18/190
Classifying difficulty for 19/190
Classifying difficulty for 20/190
Classifying difficulty for 21/190
Classifying difficulty for 22/190
Classifying difficulty for 23/190
Classifying difficulty for 24/190
Classifying difficulty for 25/190
Classifying difficulty for 26/190
Classifying difficulty for 27/190
Classifying difficulty for 28/190
Classifying difficulty for 29/190
Classifying difficulty 

In [31]:
cols = ['image_id', 'lat', 'lon', 'scene_type', 'difficulty', 'terrain', 'dist_to_water_km',
        'water_pct', 'wetland_pct', 'urban_pct', 'forest_pct', 'agriculture_pct',
        'open_vegetation_pct', 'bare_pct', 'snow_pct', 'vegetation_pct', 'cloud_category']

df = df[cols]

In [ ]:
df.to_csv("dataset_full.csv", index=False)

In [33]:
scene_difficulty_matrix = pd.crosstab(
    df["scene_type"],
    df["difficulty"],
    margins=True,
    margins_name="Total"
).reindex(columns=["Easy", "Medium", "Hard", "Extreme", "Total"], fill_value=0)

scene_difficulty_matrix.T

scene_type,Agriculture,Coastline,Desert,Forest,Mixed,Mountain,Ocean,Urban,Total
difficulty,,,,,,,,,
Easy,17,16,0,18,19,0,0,18,88
Medium,2,10,7,3,5,15,0,2,44
Hard,0,1,7,0,0,11,17,2,38
Extreme,1,3,1,0,4,4,7,0,20
Total,20,30,15,21,28,30,24,22,190
